# Summary
## Done
- Loaded 48 csv_files from ../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv
- Reading csv's and checking for inconsistent schema (i.e. heading/feature names switching places between csv files)
- Found 11 csv files that have misaligned headers/features
- Checked all heading/feature names (Number, Start date, etc.) across all 48 files are spelled exactly the same - i.e. no mis-spellings like "Nmber".
- Header/feature names are consistent across all 48 .csv files now. EDA will be performed seperately to see if any further cleaning is necessary.

In [1]:
import pandas as pd
import os
import shutil
from pathlib import Path

In [2]:
data_dir = Path("../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv")
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} files in {data_dir}")

Found 48 files in ../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv


## Checking header and data consistency across thes 48 csv files

In [3]:
#Checking all headers
print(f"Listing ALL {len(csv_files)} csv files")
print()
for f in csv_files:
    #Only loading first few rows to reduce load on cpu
    df = pd.read_csv(f, nrows=0) 
    print(f"File: {f.name}")
    print(list(df.columns))
    print("-" * 160)
pd.set_option('display.max_columns', None) 

Listing ALL 48 csv files

File: 387JourneyDataExtract01Jan2024-14Jan2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 388JourneyDataExtract15Jan2024-31Jan2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 389JourneyDataExtract01Feb2024-14Feb2024.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number',

Some **column headings are misaligned** or in the wrong place. I'll have to check each file to make sure the data isn't in the wrong columns too.

After manual inspection, most csv files have this schema:
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']

We'll call this list: **correct_headers**

### CORRECT HEADERS: 

<font size="4" color="green">['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']</font>

In [4]:
#Checking which csv files have differing header order
print("PROBLEM FILES")
#The majority of the .csv files have this schema:
correct_headers = ['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
problem_files = []
print(f"\033[32mCORRECT HEADERS: \n{correct_headers}\033[0m") #green font to stand out
print("-" * 160)
print("-" * 160)
for f in csv_files:
    df = pd.read_csv(f, nrows=0)  
    if (list(df.columns)) != correct_headers:
        print(f"File: {f.name}")
        print(list(df.columns))
        print("-" * 160)
        problem_files.append(f.name)

PROBLEM FILES
CORRECT HEADERS: 
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 423JourneyDataExtract01Jul2025-15Jul2025.csv
['Number', 'Start date', 'Start station', 'Start station number', 'End date', 'End station', 'End station number', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 424JourneyDataExtract16Jul2025-31Jul2025.csv
['Number', 'S

In [5]:
print(f"Number of inconsistent files: {len(problem_files)}")

Number of inconsistent files: 11


We have 11 csv files with **inconsistent schema**. For each of these 11 csv files, we will need to realign their columns to match the **correct_headers**

Need to check if these csv files have the same feature names as the Correct headers list above. All header names must match for all 48 files.


In [6]:
#Need to check if these csv files have the same feature names as the Correct headers list above. All header names must match for all 48 files.
expected = set(correct_headers)
incon_files = []
for f in csv_files:
    df = pd.read_csv(f, nrows=0)
    if (list(df.columns)) != correct_headers:
        print(f"File {f.name} has same Column headings? {set(df.columns) == expected}")
        incon_files.append(f.name) #This will help when correcting column headings 
        print

File 423JourneyDataExtract01Jul2025-15Jul2025.csv has same Column headings? True
File 424JourneyDataExtract16Jul2025-31Jul2025.csv has same Column headings? True
File 425JourneyDataExtract01Aug2025-15Aug2025.csv has same Column headings? True
File 427JourneyDataExtract01Sep2025-15Sep2025.csv has same Column headings? True
File 428JourneyDataExtract16Sep2025-30Sep2025.csv has same Column headings? True
File 429JourneyDataExtract01Oct2025-15Oct2025.csv has same Column headings? True
File 430JourneyDataExtract16Oct2025-31Oct2025.csv has same Column headings? True
File 431JourneyDataExtract01Nov2025-15Nov2025.csv has same Column headings? True
File 432JourneyDataExtract16Nov2025-30Nov2025.csv has same Column headings? True
File 433JourneyDataExtract01Dec2025-15Dec2025.csv has same Column headings? True
File 434JourneyDataExtract16Dec2025-31Dec2025.csv has same Column headings? True


Because the headers have the same names and are just in a diffrent order, I can now re-order headers and write them as corrected csv's (NOT overwriting original files)

In [7]:
incon_files

['423JourneyDataExtract01Jul2025-15Jul2025.csv',
 '424JourneyDataExtract16Jul2025-31Jul2025.csv',
 '425JourneyDataExtract01Aug2025-15Aug2025.csv',
 '427JourneyDataExtract01Sep2025-15Sep2025.csv',
 '428JourneyDataExtract16Sep2025-30Sep2025.csv',
 '429JourneyDataExtract01Oct2025-15Oct2025.csv',
 '430JourneyDataExtract16Oct2025-31Oct2025.csv',
 '431JourneyDataExtract01Nov2025-15Nov2025.csv',
 '432JourneyDataExtract16Nov2025-30Nov2025.csv',
 '433JourneyDataExtract01Dec2025-15Dec2025.csv',
 '434JourneyDataExtract16Dec2025-31Dec2025.csv']

## Saving csv's with correct headers & Moving all csv's to new Clean Folder

In [8]:
src = '../data/tfl-cycling-journey-data-2024-2025/raw-2024-2025-csv'
dst = '../data/tfl-cycling-journey-data-2024-2025/clean-2024-2025-csv/'

for f in csv_files:
    if f.name in incon_files:
        df = pd.read_csv(f) #Reading in entire csv file this time
        #display(df.head(2))
        df_reordered = df[correct_headers] #set headers in current dataframe to match correct_headers
        #display(df_reordered.head(2))
        df_reordered.to_csv(f'{dst}{f.name}', index=False, encoding='utf-8')
        print(f"File: {f.name} Saved!")
        print()
    else:
        #Copy all other files from the raw-2024-2025-csv to clean-2024-2025-csv
        src_path = os.path.join(src, f"{f.name}")
        shutil.copy(src_path, dst)
        print(f"File: {f.name} Copied!")
        print()


Correct Headers:
 ['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
File: 387JourneyDataExtract01Jan2024-14Jan2024.csv Copied!

File: 388JourneyDataExtract15Jan2024-31Jan2024.csv Copied!

File: 389JourneyDataExtract01Feb2024-14Feb2024.csv Copied!

File: 390JourneyDataExtract15Feb2024-29Feb2024.csv Copied!

File: 391JourneyDataExtract01Mar2024-14Mar2024.csv Copied!

File: 392JourneyDataExtract15Mar2024-31Mar2024.csv Copied!

File: 393JourneyDataExtract01Apr2024-14Apr2024.csv Copied!

File: 394JourneyDataExtract15Apr2024-30Apr2024.csv Copied!

File: 395JourneyDataExtract01May2024-14May2024.csv Copied!

File: 396JourneyDataExtract15May2024-31May2024.csv Copied!

File: 397Journey

## Quick test to see if All headers are conistent

In [14]:
clean_data_dir = Path("../data/tfl-cycling-journey-data-2024-2025/clean-2024-2025-csv")
clean_csv_files = sorted(clean_data_dir.glob("*.csv"))
print(f"Found {len(clean_csv_files)} files in {clean_data_dir}")

Found 48 files in ../data/tfl-cycling-journey-data-2024-2025/clean-2024-2025-csv


In [16]:
print(f"\033[32mCORRECT HEADERS: \n{correct_headers}\033[0m") #green font to stand out
print("-" * 160)
print("-" * 160)
for f in clean_csv_files:
    df = pd.read_csv(f, nrows=0)  
    if (list(df.columns)) != correct_headers:
        print(f"File: {f.name}")
        print(list(df.columns))
        print("-" * 160)
    else:
        print(f"File ({f.name}) is OKAY!")

CORRECT HEADERS: 
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------------------------------------------------------------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------------------------------------------------------------------
File (387JourneyDataExtract01Jan2024-14Jan2024.csv) is OKAY!
File (388JourneyDataExtract15Jan2024-31Jan2024.csv) is OKAY!
File (389JourneyDataExtract01Feb2024-14Feb2024.csv) is OKAY!
File (390JourneyDataExtract15Feb2024-29Feb2024.csv) is OKAY!
File (391JourneyDataExtract01Mar2024-14Mar2024.csv) is OKAY!
File (392JourneyDataExtract15Mar2024-31Mar2024.csv) is OKAY!
File (393JourneyDataExtract01Apr2024-14Apr2024.csv) is OKAY!
File (394JourneyDataExtract15Apr2024-30Apr2024.c